# Bisect: train on the original `anchors.py`

Everything is held at run 3's settings. The only thing that changes is
where the anchor worlds come from: the original `anchors.py` instead of
`anchors_v22.py`. Training code, system prompt, optimizer, example count
and seeds are identical to run 3.

Training and evaluation are in one notebook, because the evaluation here
is localization only: 32 prompts at 64 tokens each, about three minutes.
No second dataset round trip.

## What this decides

| run | anchors | steps | arm_c localize | detect sens | preservation |
| --- | --- | --- | --- | --- | --- |
| August | bespoke `anchors.py` | 105 | **31/32** | 0/64 | constant 64/64 |
| run 1 | official gen, mixed | 66 | 5/32 | 31/32 | 11/64 |
| run 2 | official gen, single cond | 66 | 4/32 | 30/32 | 7/64 |
| run 3 | official gen, single cond | 105 | 7/32 | 0/32 | constant 64/64 |
| this run | **bespoke `anchors.py`** | 105 | ? | | |

Run 3 reproduced two of August's three signatures exactly: detection
false on everything, preservation constant on everything. Only
localization did not come back. So `PRES_REPEAT` and `GRAD_ACCUM` account
for those two behaviours, and whatever carries localization is in the
anchor worlds themselves.

* **Near 31/32** -> the effect is a property of the bespoke worlds. They
  are all `E -> F` with the goal absorbing behind exactly one inbound
  edge, where the official generator varies start and goal across 19
  pairs. That would say what the model was actually learning, which is a
  better result than the original claim.
* **Near 7/32** -> 31/32 does not reproduce from its own recipe, and the
  writeup is a fragility finding with four controls behind it.

In [8]:
import os
# Trainer wraps the model in DataParallel when it sees two devices, which
# halves the step count and doubles the effective batch. Pin to one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [9]:
%pip install -q -U transformers peft bitsandbytes accelerate

Note: you may need to restart the kernel to use updated packages.


## Preflight

In [10]:
import sys, glob, json, time, importlib.util, collections
import torch

def find_dir(marker, root="/kaggle/input"):
    hits = sorted(glob.glob(os.path.join(root, "**", marker), recursive=True),
                  key=lambda p: (p.count(os.sep), len(p)))
    if not hits:
        raise SystemExit(f"no {marker} under {root}")
    return os.path.dirname(hits[0])

RUN_TAG = "origanchors"
REPO_PATH = find_dir("resource_mdp.py")
EVAL_PATH = find_dir("anchors_v22.py")

_old = sorted(glob.glob("/kaggle/input/**/anchors.py", recursive=True))
_old = [p for p in _old if os.path.basename(p) == "anchors.py"]
if not _old:
    raise SystemExit(
        "anchors.py (the original) is not attached. Add it to the ecpm eval "
        "dataset or upload it separately. It is the file in your ecpm/ "
        "folder with START, GOAL = \"E\", \"F\" near the top.")
OLD_ANCHORS = _old[0]
PAY_CHANGED = sorted(glob.glob("/kaggle/input/**/payloads_silent_break_det",
                               recursive=True))[0]
OUT_DIR = "/kaggle/working"
print("repo:        ", REPO_PATH)
print("eval:        ", EVAL_PATH)
print("old anchors: ", OLD_ANCHORS)
print("payloads:    ", PAY_CHANGED)

assert torch.cuda.is_available(), "no GPU: set Accelerator in the sidebar"
GPU = torch.cuda.get_device_name(0)
CAP = torch.cuda.get_device_capability(0)
USE_BF16 = CAP[0] >= 8            # T4 is 7.5 and only emulates bf16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"\ngpu: {GPU} | capability {CAP[0]}.{CAP[1]} | dtype: {DTYPE}")
print("visible devices:", torch.cuda.device_count(), "(must be 1)")
assert torch.cuda.device_count() == 1, (
    "CUDA_VISIBLE_DEVICES did not take; restart the kernel and run from the "
    "first cell")

import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

repo:         /kaggle/input/datasets/mazwyy/ecpm-repo/ecpm-efe
eval:         /kaggle/input/datasets/mazwyy/ecpm-eval
old anchors:  /kaggle/input/datasets/mazwyy/ecpm-old-anchors/anchors.py
payloads:     /kaggle/input/datasets/mazwyy/ecpm-payloads/payloads_silent_break_det

gpu: Tesla T4 | capability 7.5 | dtype: torch.float16
visible devices: 1 (must be 1)
transformers 5.17.0 | peft 0.20.0


In [11]:
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_OUT = f"{OUT_DIR}/anchor_adapter_{RUN_TAG}_{MODEL_NAME.split('/')[-1]}"

N_WORLDS   = 40          # 140 items, exactly August
K          = 5
TOTAL_STEPS = 105        # 140 // 4 * 3, run 3's count
LR         = 1e-4
GRAD_ACCUM = 4
MAX_LEN    = 2048
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.0
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
           "gate_proj", "up_proj", "down_proj"]

sys.path.insert(0, EVAL_PATH)
import ecpm_eval as E
E.attach(REPO_PATH)

spec = importlib.util.spec_from_file_location("old_anchors", OLD_ANCHORS)
oa = importlib.util.module_from_spec(spec)
spec.loader.exec_module(oa)
print("loaded the original anchors.py |", oa.NODES,
      "| start/goal", oa.START, oa.GOAL)

ecpm_parser attached from /kaggle/input/datasets/mazwyy/ecpm-repo/ecpm-efe (looks like v2.2)
loaded the original anchors.py | ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H'] | start/goal E F


## Build the anchor set from the original generator

`build_anchors` returns worlds carrying their own evidence and a list of
`(probe, question, answer)` triples. August trained on
`evidence + "\n\n" + question` as the user turn with the answer as the
assistant turn, which is what is reproduced here.

In [12]:
worlds = oa.build_anchors(n_worlds=N_WORLDS, k=K)
examples = [{"seed": w["seed"], "probe": p,
             "prompt": w["evidence"] + "\n\n" + q, "gold": a}
            for w in worlds for p, q, a in w["items"]]

print(f"worlds {len(worlds)} | changed {sum(w['changed'] for w in worlds)} "
      f"| examples {len(examples)}")
print("by probe:", dict(collections.Counter(e["probe"] for e in examples)))
print("seed range:", min(w["seed"] for w in worlds), "-",
      max(w["seed"] for w in worlds))
print("distinct start/goal: 1 (this generator is always E -> F)")
assert len(examples) == 140, f"expected August's 140 examples, got {len(examples)}"
assert min(w["seed"] for w in worlds) > 79, "anchor seeds must clear 0-79"
json.dump([{k: v for k, v in w.items() if k != "items"} for w in worlds],
          open(f"{OUT_DIR}/anchor_worlds_{RUN_TAG}.json", "w"), default=str)

worlds 40 | changed 20 | examples 140
by probe: {'detection': 40, 'preservation': 40, 'adaptation': 40, 'localization': 20}
seed range: 1000 - 1040
distinct start/goal: 1 (this generator is always E -> F)


## Tokenize

Same masking and the same `ecpm_eval.SYSTEM` as run 3, so the anchor data
is the only variable.

In [13]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def as_ids(x):
    # apply_chat_template may return a BatchEncoding, which subclasses
    # UserDict rather than dict, so list(x) would give the keys
    if hasattr(x, "input_ids"):
        x = x.input_ids
    elif hasattr(x, "keys") and "input_ids" in x.keys():
        x = x["input_ids"]
    x = list(x)
    if x and isinstance(x[0], (list, tuple)):
        x = list(x[0])
    if not all(isinstance(t, int) for t in x):
        raise TypeError(f"expected token ids, got {type(x[0]).__name__}")
    return x

def encode(ex):
    prefix = as_ids(tok.apply_chat_template(
        [{"role": "system", "content": E.SYSTEM},
         {"role": "user", "content": ex["prompt"]}],
        add_generation_prompt=True, tokenize=True))
    ans = as_ids(tok(ex["gold"] + tok.eos_token,
                     add_special_tokens=False)["input_ids"])
    return {"input_ids": prefix + ans,
            "labels": [-100] * len(prefix) + ans,
            "n_prefix": len(prefix), "n_answer": len(ans)}

enc = [encode(e) for e in examples]
lengths = sorted(len(x["input_ids"]) for x in enc)
prefixes = sorted(x["n_prefix"] for x in enc)
print(f"{len(enc)} examples | tokens min {lengths[0]} "
      f"median {lengths[len(lengths)//2]} max {lengths[-1]}")
print(f"prefix tokens min {prefixes[0]} max {prefixes[-1]}")
assert lengths[-1] <= MAX_LEN, f"{lengths[-1]} > MAX_LEN"
assert prefixes[0] > 300, "chat template did not tokenize properly"
assert all(any(l != -100 for l in x["labels"]) for x in enc)

140 examples | tokens min 1388 median 1484 max 1520
prefix tokens min 1382 max 1443


In [14]:
from torch.utils.data import Dataset

class Anchors(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        return {"input_ids": r["input_ids"], "labels": r["labels"]}

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        gap = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad] * gap)
        out["labels"].append(b["labels"] + [-100] * gap)
        out["attention_mask"].append([1] * len(b["input_ids"]) + [0] * gap)
    return {k: torch.tensor(v) for k, v in out.items()}

train_ds = Anchors(enc)
print(len(train_ds), "training examples")

140 training examples


## Train

In [15]:
from transformers import (AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=DTYPE,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM", target_modules=TARGETS))
model.print_trainable_parameters()

print(f"\nmax_steps={TOTAL_STEPS}, grad_accum={GRAD_ACCUM} "
      f"-> effective batch {GRAD_ACCUM}, same as run 3")
result = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=f"{OUT_DIR}/bisect_tmp",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        max_steps=TOTAL_STEPS, learning_rate=LR,
        warmup_steps=max(1, TOTAL_STEPS // 20),
        lr_scheduler_type="cosine", logging_steps=10,
        save_strategy="no", report_to=[],
        bf16=USE_BF16, fp16=not USE_BF16,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False}),
    train_dataset=train_ds, data_collator=collate).train()
print(result.metrics)
print("\nAugust final loss 0.049 | run 2 0.104 | run 3 see its provenance")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

max_steps=105, grad_accum=4 -> effective batch 4, same as run 3


Step,Training Loss
10,0.094248
20,0.053959
30,0.041964
40,0.052525
50,0.036584
60,0.029596
70,0.026748
80,0.030235
90,0.025130
100,0.023552


{'train_runtime': 1017.6767, 'train_samples_per_second': 0.413, 'train_steps_per_second': 0.103, 'total_flos': 4875952995634176.0, 'train_loss': 0.040697846242359705, 'epoch': 3.0}

August final loss 0.049 | run 2 0.104 | run 3 see its provenance


In [16]:
model.save_pretrained(ADAPTER_OUT)
tok.save_pretrained(ADAPTER_OUT)
json.dump({"model": MODEL_NAME, "phase": 1, "run": "original_anchors_bisect",
           "anchor_source": "anchors.py (bespoke, E->F only)",
           "anchor_worlds": len(worlds), "anchor_examples": len(examples),
           "optimizer_steps": TOTAL_STEPS, "lr": LR, "grad_accum": GRAD_ACCUM,
           "lora": {"r": LORA_R, "alpha": LORA_ALPHA,
                    "dropout": LORA_DROPOUT, "targets": TARGETS},
           "final_loss": result.metrics.get("train_loss")},
          open(f"{ADAPTER_OUT}/phase1_provenance.json", "w"), indent=1)
print("saved to", ADAPTER_OUT)

saved to /kaggle/working/anchor_adapter_origanchors_Qwen2.5-1.5B-Instruct


## Evaluate, localization only

32 prompts at 64 tokens. The other three probes are skipped: run 3
already showed detection and preservation reproduce, and localization is
the only number in question.

In [17]:
import pandas as pd

payloads = E.load_payloads(PAY_CHANGED)
seeds = sorted(p["seed"] for p in payloads)[:32]
payloads = [p for p in payloads if p["seed"] in seeds]
print(f"{len(payloads)} seeds: {seeds}")

model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

def generate(messages, max_new_tokens):
    e = tok.apply_chat_template(messages, add_generation_prompt=True,
                                return_tensors="pt", return_dict=True)
    e = {k: v.to(model.device) for k, v in e.items()}
    with torch.no_grad():
        o = model.generate(**e, max_new_tokens=max_new_tokens,
                           do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(o[0, e["input_ids"].shape[1]:], skip_special_tokens=True)

t0 = time.time()
rows = E.run_arm(payloads, generate, arm="arm_c", mode="single",
                 probes=["localization"],
                 out_path=f"{OUT_DIR}/raw_{RUN_TAG}_arm_c.jsonl")
print(f"done in {(time.time()-t0)/60:.1f} min")

ok = [r["seed"] for r in rows if r["scored"].get("correct")]
node = sum(1 for r in rows if r["parsed"].get("node") == r["target"][0])
print(f"\narm_c localization {len(ok)}/{len(rows)}  node-level {node}/{len(rows)}")
print("correct on seeds:", sorted(ok))
print("August was 31/32, missing only seed 23")

32 seeds: [0, 1, 4, 5, 7, 8, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 28, 29, 30, 33, 34, 35, 36, 37, 38, 39, 40]
  5/32 payloads
  10/32 payloads
  15/32 payloads
  20/32 payloads
  25/32 payloads
  30/32 payloads
done in 1.1 min

arm_c localization 32/32  node-level 32/32
correct on seeds: [0, 1, 4, 5, 7, 8, 9, 10, 11, 12, 13, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 28, 29, 30, 33, 34, 35, 36, 37, 38, 39, 40]
August was 31/32, missing only seed 23


In [18]:
print(f"{'run':26} {'anchors':26} {'steps':6} {'localize'}")
for name, anch, steps, loc in (
        ("August", "bespoke anchors.py", 105, "31/32"),
        ("run 1 mixed", "official gen, 4 conditions", 66, "5/32"),
        ("run 2 singlecond", "official gen, silent_break", 66, "4/32"),
        ("run 3 augustopt", "official gen, silent_break", 105, "7/32"),
        ("this bisect", "bespoke anchors.py", TOTAL_STEPS,
         f"{len(ok)}/{len(rows)}")):
    print(f"  {name:24} {anch:26} {steps:<6} {loc}")

df = pd.DataFrame([{"seed": r["seed"], "gold": " ".join(r["target"]),
                    "said": (f"{r['parsed'].get('node')} "
                             f"{r['parsed'].get('action')}"
                             if r["parsed"]["status"] == "ok"
                             else r["parsed"]["status"]),
                    "correct": bool(r["scored"].get("correct"))}
                   for r in sorted(rows, key=lambda r: r["seed"])])
display(df)

run                        anchors                    steps  localize
  August                   bespoke anchors.py         105    31/32
  run 1 mixed              official gen, 4 conditions 66     5/32
  run 2 singlecond         official gen, silent_break 66     4/32
  run 3 augustopt          official gen, silent_break 105    7/32
  this bisect              bespoke anchors.py         105    32/32


,seed,gold,said,correct
0,0,E a1,E a1,True
1,1,A a2,A a2,True
2,4,E a3,E a3,True
3,5,F a2,F a2,True
4,7,D a2,D a2,True
5,8,G a1,G a1,True
6,9,C a3,C a3,True
7,10,F a2,F a2,True
8,11,F a2,F a2,True
9,12,F a1,F a1,True


## Package

In [19]:
import zipfile, shutil
shutil.rmtree(f"{OUT_DIR}/bisect_tmp", ignore_errors=True)
ZIP_PATH = f"{OUT_DIR}/{RUN_TAG}_all.zip"
n = 0
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(glob.glob(f"{OUT_DIR}/**/*", recursive=True)):
        if not os.path.isfile(path) or os.path.basename(ZIP_PATH) in path:
            continue
        if ".ipynb_checkpoints" in path:
            continue
        z.write(path, os.path.relpath(path, OUT_DIR))
        n += 1
print(f"{n} files -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)")

9 files -> /kaggle/working/origanchors_all.zip (70.4 MB)
